# Multiple Linear Regression — Course Reconstruction + Modern Python

This notebook captures the next course section and corrects outdated APIs while keeping the original learning intent.

Core idea: simple regression has one feature. Multiple regression keeps the same idea but gives **each feature its own coefficient**.


## 1. 📘 Theory

Simple linear regression:

\[
\hat{Y}=mX+c
\]

Multiple linear regression:

\[
\hat{Y}=b_0+b_1X_1+b_2X_2+\cdots+b_pX_p
\]

Course example:
- R&D Spend
- Administration
- Marketing Spend
- State
→ predict **Profit**

Each numerical feature gets its own learned coefficient. Categorical features must be represented numerically before a standard linear model can use them.


## 2. 🧠 Intuition

Simple regression asks:

> How does one input relate to Y?

Multiple regression asks:

> Holding the other included features fixed, how is each feature associated with Y?

A coefficient is not automatically a causal effect. Correlated features, omitted variables, interactions, data leakage, and poor data quality can all change interpretation.


## 3. 🏗️ Data-engineering mental model

Think of a feature matrix as a curated analytical table:

```text
one row = one company / observation
X1      = R&D Spend
X2      = Administration
X3      = Marketing Spend
X4...   = encoded State columns
Y       = Profit
```

Your data-engineering work prepares the trustworthy feature table; ML learns patterns from that table.


## 4. 💻 Libraries introduced

The course introduced:
- NumPy (`np`)
- Pandas (`pd`)
- Matplotlib (`plt`)
- Seaborn (`sns`)
- scikit-learn

Correction: `sns` refers to **Seaborn**, not Pandas. Seaborn can consume Pandas DataFrames conveniently.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 5. 📘 Load and inspect data

Typical pattern:

```python
companies = pd.read_csv("1000_Companies.csv")
companies.head()
```

`head()` shows the first rows so we can inspect schema, values, categorical columns, missingness clues, and whether the target looks correct.

Do not jump directly to model training before understanding the dataset.


In [ ]:
# Example only: replace with a public/synthetic CSV when practicing.
# companies = pd.read_csv("1000_Companies.csv")
# display(companies.head())


## 6. 🧠 X and y

In tabular supervised learning:
- `X` usually means the **feature matrix**: many rows × many input columns
- `y` usually means the **target vector**: one target value per row

Convention: capital `X` reflects a matrix; lowercase `y` reflects a vector. It is convention, not a Python requirement.


In [ ]:
# Modern readable Pandas style:
# X = companies.drop(columns=["Profit"])
# y = companies["Profit"]

# Course-style iloc idea:
# X = companies.iloc[:, :-1]
# y = companies.iloc[:, -1]


## 7. 📊 Correlation heatmap

A correlation matrix can be a useful **first diagnostic** for numerical variables.

Important:
- correlation is not causation
- correlation mainly describes linear association
- high correlation between predictors can create multicollinearity
- categorical columns are not directly included unless encoded
- a heatmap is exploratory, not proof that a feature belongs in the final model


In [ ]:
# Example:
# numeric_corr = companies.corr(numeric_only=True)
# sns.heatmap(numeric_corr, annot=True)
# plt.title("Correlation matrix")
# plt.show()


## 8. 📘 Categorical variables and one-hot encoding

The course used `LabelEncoder` followed by an older `OneHotEncoder(categorical_features=...)` pattern.

That API is outdated.

For an unordered category such as State, assigning:
`California=0, Florida=1, New York=2`
can accidentally suggest an order/distance that does not exist.

**One-hot encoding** creates separate indicator columns instead.


In [ ]:
# Modern scikit-learn approach:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Example only:
# categorical_features = ["State"]
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("state", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
#     ],
#     remainder="passthrough"
# )


## 9. 🧠 Dummy-variable trap

If a categorical feature has k categories and we include all k indicator columns **plus an intercept**, one column can be perfectly reconstructed from the others.

Dropping one reference category is a common way to avoid perfect multicollinearity and make coefficients easier to interpret.

Modern tools may still compute a solution using numerical linear algebra, but the underlying identifiability issue is real.


## 10. 📘 Train/test split

The course used:

```python
train_test_split(X, y, test_size=0.2, random_state=0)
```

Meaning:
- 80% → training
- 20% → testing
- `random_state=0` makes the random split **reproducible**

It does not mean “randomize differently every time.” The seed makes the split repeatable.


In [ ]:
from sklearn.model_selection import train_test_split

# Example after preprocessing-ready data exists:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=0
# )


## 11. 🧠 Why split at all?

If we evaluate only on the same data used to fit the model, the score can be misleading.

Training set:
> learn parameters

Test set:
> simulate unseen data

This is the beginning of **generalization thinking** — performance on data the model did not train on.


## 12. ⚡ Fit the model

The course's most important library call:

```python
model.fit(X_train, y_train)
```

Mental translation:

```text
training examples
    ↓
solve for coefficients + intercept
    ↓
minimize the linear model's least-squares objective
    ↓
store learned parameters
```

The math did not disappear. The library performs it for us.


In [ ]:
from sklearn.linear_model import LinearRegression

# regressor = LinearRegression()
# regressor.fit(X_train, y_train)


## 13. ⚡ Predict

```python
y_pred = regressor.predict(X_test)
```

For every held-out feature row, the fitted model applies its learned equation and returns a predicted target.


In [ ]:
# y_pred = regressor.predict(X_test)
# print(y_pred[:10])


## 14. 🔢 Coefficients and intercept

Correct scikit-learn attributes:

```python
regressor.coef_
regressor.intercept_
```

The course narration referred to “coefficient” in a few different ways. Keep these concepts separate:

- `coef_` → one coefficient per encoded input feature
- `intercept_` → baseline constant term
- together they define the learned linear equation


In [ ]:
# print("coefficients:", regressor.coef_)
# print("intercept:", regressor.intercept_)


## 15. ⚠️ Why coefficient interpretation can become hard

A coefficient means the model's estimated change in Y for a one-unit feature change **while other included features are held constant**.

Interpretation becomes tricky when:
- features are strongly correlated
- features use very different scales
- categories are encoded
- interactions/nonlinearity exist
- important variables are omitted
- the dataset is observational rather than experimental


## 16. 💊 Pharma application lens

Synthetic/public multiple-regression example:

```text
historical demand
marketing/activity indicator
territory size
seasonality
public demographic feature
        ↓
continuous product-demand estimate
```

The value is not the exact dataset; it is learning how a multi-feature model behaves and how to validate it responsibly.


## 17. 🌱 PlantMind application lens

Possible synthetic features:
- vibration
- temperature
- pressure
- load
- operating hours
- physics-derived stress index
→ continuous health/degradation score

This naturally moves beyond the one-feature example.


## 18. 🧪 Exercise

Build a tiny synthetic dataset with:
- `rd_spend`
- `admin_spend`
- `marketing_spend`
- `state`
- `profit`

Then:
1. inspect it with Pandas
2. identify X/y
3. one-hot encode state
4. split train/test
5. fit LinearRegression
6. predict
7. inspect coefficients
8. calculate evaluation metrics

Do not copy the course's outdated encoding API.


## 19. ✅ Explain-back checkpoint

You have understood multiple regression when you can explain:

> It is the same linear-model idea as simple regression, but X is now a matrix of several features and the model learns one coefficient per feature plus an intercept. We still need clean features, held-out evaluation, and cautious interpretation.


## 20. ❓ Questions / gaps
- Why does one-hot encoding increase the number of columns?
- What exactly does “holding other features constant” mean?
- What happens when two predictors are nearly duplicates?
- Why can a high train score still be bad?
- When should linear regression be replaced by a nonlinear model?
